# Python GIS Procedural Pipeline

This notebook implements an end-to-end pipeline:
1. Draw a polygon on an interactive map
2. Extract GeoJSON & transform coordinates to UTM (CCW ordering)
3. Generate a procedural building with PyPRT (CityEngine)
4. Visualize the 3D model with Autodesk Viewer

## تثبيت المكتبات (Install Libraries)

If running outside the project venv, uncomment the following line:
```
# !pip install leafmap shapely pyproj pyprt pythreejs ipywidgets trimesh
```

---
## الخلية 1: إعداد الخريطة ورسم المضلع
## Cell 1: Interactive Map & Polygon Drawing

In [ ]:
import leafmap

m = leafmap.Map(center=[15, 45], zoom=16)

# The draw tool is available by default in leafmap.
# Draw a polygon on the map below, then proceed to Cell 2.
m

---
## الخلية 2: استخراج GeoJSON وتحويل الإحداثيات إلى UTM وترتيب CCW
## Cell 2: Extract GeoJSON → Transform to UTM → Ensure CCW Winding

In [ ]:
from shapely.geometry import shape, Polygon
from pyproj import Transformer

if len(m.draw_features) == 0:
    raise RuntimeError("لم يتم رسم أي مضلع بعد — No polygon drawn yet. Run Cell 1 and draw a polygon first.")

# Take the most recently drawn feature
geojson_feature = m.draw_features[-1]
polygon = shape(geojson_feature['geometry'])
coords = list(polygon.exterior.coords)

print("إحداثيات Lat/Lon الأصلية (Original Lat/Lon coordinates):")
print(coords)

# Transform from WGS84 (EPSG:4326) to UTM Zone 38N (EPSG:32638)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32638", always_xy=True)
utm_coords = [transformer.transform(x, y) for x, y in coords]
utm_polygon = Polygon(utm_coords)

# Ensure counter-clockwise (CCW) winding order
if not utm_polygon.exterior.is_ccw:
    utm_coords = list(utm_polygon.exterior.coords)[::-1]
    utm_polygon = Polygon(utm_coords)

print("\nتم تحويل الإحداثيات إلى UTM وترتيبها CCW بنجاح")
print("(Coordinates transformed to UTM and ordered CCW successfully)")
print(f"\nUTM Coords ({len(utm_coords)} vertices):")
for i, c in enumerate(utm_coords):
    print(f"  [{i}] x={c[0]:.2f}, y={c[1]:.2f}")

from IPython.display import display
display(utm_polygon)  # Show the 2D polygon shape

# Create an interactive map for the extracted feature
m2 = leafmap.Map(center=[15, 45], zoom=16)
fc = {"type": "FeatureCollection", "features": [geojson_feature]}
m2.add_geojson(fc, layer_name="Extracted Polygon")
m2

---
## الخلية 3: إرسال الشكل إلى PyPRT وتوليد المبنى Procedural
## Cell 3: Send Shape to PyPRT & Generate Procedural Building

In [ ]:
import pyprt
import os
import numpy as np
import trimesh
import json

# ── Verify Cell 2 output exists ──────────────────────────────────────────────
try:
    utm_polygon
except NameError:
    raise RuntimeError("لم يتم تعريف المضلع. تأكد من تنفيذ الخلية السابقة.")

# ── Initialize PRT engine ────────────────────────────────────────────────────
pyprt.initialize_prt()

# ── Build flat vertex list [x0,y0,z0, x1,y1,z1, ...] ────────────────────────
coords_2d = list(utm_polygon.exterior.coords)[:-1] 
flat_coords = []
for x, y in coords_2d:
    flat_coords.extend([x, 0.0, y]) 

# ── Create Initial Shape ─────────────────────────────────────────────────────
initial_shape = pyprt.InitialShape(flat_coords)

# ── Building attributes ──────────────────────────────────────────────────────
attributes = {
    "Usage":                              "Residential",
    "Mode":                               "Generate Facade",
    "Nbr_of_Floors":                      8,
    "Standard_Floor_Height":               3.200,
    "Ground_Floor_Height":                 5.000,
    "Front_Setback_Mode":                 "Increasing",
    "Front_Setback_Distance":              4.500,
    "Layout_Shape":                       "Along Front",
    "Wing_Width":                          15.000,
    "Layout_Orientation":                  "Open To Back",
    "Green_Space.Generate_Green_Space":     True,
    "Green_Space.Create_Trees":             False,
}

# ── Rule Package path ────────────────────────────────────────────────────────
RPK_PATH = r"C:\RPK\RuleFootprint.rpk"
if not os.path.isfile(RPK_PATH):
    print(f"⚠️  RPK file not found at: {RPK_PATH}")
else:
    print(f"✅ RPK found: {RPK_PATH}")

# ── Generate the model ───────────────────────────────────────────────────────
model_generator = pyprt.ModelGenerator([initial_shape])
generated_models = model_generator.generate_model(
    [attributes], RPK_PATH, "com.esri.pyprt.PyEncoder",
    {"emitGeometry": True, "emitReport": True}
)

if generated_models and generated_models[0]:
    gm = generated_models[0]
    raw_verts = np.array(gm.get_vertices(), dtype=np.float32).reshape(-1, 3)
    
    # ── CRITICAL: CENTER THE MODEL (Recenter to 0,0,0) ────────────────────────
    center_point = np.mean(raw_verts, axis=0)
    centered_verts = raw_verts - center_point
    print(f"Recenter: Model moved from {center_point} to origin.")
    
    raw_indices = gm.get_indices()
    raw_faces = gm.get_faces()
    triangles = []
    offset = 0
    for n in raw_faces:
        face_indices = raw_indices[offset:offset + n]
        for i in range(1, n - 1):
            triangles.append([face_indices[0], face_indices[i], face_indices[i + 1]])
        offset += n

    mesh = trimesh.Trimesh(vertices=centered_verts, faces=triangles)
    gltf_data = mesh.export(file_type='gltf')
    if isinstance(gltf_data, bytes): gltf_data = gltf_data.decode('utf-8')
    gltf_list = [json.loads(gltf_data) if isinstance(gltf_data, str) else gltf_data]

    print(f"\n✅ Model generated and centered successfully.")
else:
    print("❌ Model generation failed.")

---
## الخلية 4: عرض النموذج باستخدام Autodesk Viewer
## Cell 4: Visualize the Model with Autodesk Viewer

In [ ]:
from IPython.display import HTML, display
import json

try:
    gltf_list
    if not gltf_list or not gltf_list[0]:
         raise Exception("gltf_list is empty.")
except NameError:
    raise Exception("Run previous cells first.")

model_json = json.dumps(gltf_list[0])

html_code = f"""
<div id='viewer-container-v8' style='position:relative; width:800px; height:600px; background: #000; border-radius: 8px;'>
    <div id='viewer' style='width:100%; height:100%'></div>
    <div id='log-v8' style='position:absolute; bottom:0; left:0; width:100%; background:rgba(0,0,0,0.8); color:#0f0; font-family:monospace; font-size:11px; padding:10px; z-index:1002;'>
        Diagnostic Viewer v8 (Fixing Error 14)...
    </div>
</div>

<link rel='stylesheet' href='https://developer.api.autodesk.com/modelderivative/v2/viewers/7.98/style.min.css'>
<script src='https://developer.api.autodesk.com/modelderivative/v2/viewers/7.98/viewer3D.min.js'></script>

<script>
function vLog(msg) {{
    var d = document.getElementById('log-v8');
    d.innerHTML += '<br>> ' + msg;
    d.scrollTop = d.scrollHeight;
}

(function() {{
    async function run() {{
        if (typeof Autodesk === 'undefined') {{
            setTimeout(run, 1000);
            return;
        }}

        Autodesk.Viewing.Initializer({{ env: 'Local', useADP: false }}, async function() {{
            var viewerDiv = document.getElementById('viewer');
            var viewer = new Autodesk.Viewing.GuiViewer3D(viewerDiv);
            viewer.start();
            
            vLog('Viewer started. Explicitly loading glTF extension...');
            
            try {{
                // Await manual extension loading as workaround for Error 14
                await viewer.loadExtension('Autodesk.glTF');
                vLog('Extension Autodesk.glTF loaded successfully.');

                var modelData = {model_json};
                var blob = new Blob([JSON.stringify(modelData)], {{type: 'application/json'}});
                var url = URL.createObjectURL(blob);
                
                var loadOptions = {{
                    fileExt: 'gltf',
                    modelName: 'model.gltf'
                }};

                vLog('Loading model from blob...');
                viewer.loadModel(url, loadOptions, function(m) {{
                    vLog('SUCCESS: Model loaded!');
                    viewer.fitToView();
                    setTimeout(function(){{ 
                        var log = document.getElementById('log-v8');
                        log.style.opacity = '0.4'; 
                        log.innerHTML = 'OK: Model Visible';
                    }}, 5000);
                }}, function(errCode, errMsg) {{
                    vLog('LOAD ERROR ' + errCode + ': ' + errMsg);
                }});

            }} catch (ex) {{
                vLog('JS EXCEPTION: ' + ex.message);
            }}
        }});
    }}
    setTimeout(run, 500);
}})();
</script>
"""
display(HTML(html_code))